# LLM-as-Judge Data Cleaning

This notebook runs the **LLM-as-judge** data cleaning pipeline that assigns quality labels
to every sample in the raw dataset.

**What the pipeline does**
- *Files with human review comments* — classifies each comment into one of nine categories:
  `blocking_issue`, `performance`, `best_practice`, `style`, `documentation`, `question`,
  `nitpick`, `praise`, `other`.
- *Files without human review comments* — asks the LLM judge whether the file contains a
  blocking issue (`true` / `false`) to verify candidate negative examples.

## 1. Imports

In [1]:
from __future__ import annotations

import json
import logging
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from ai_code_reviewer.data_processing.data_cleaning import DataCleaningPipeline
from ai_code_reviewer.data_processing.llm_client import LLMClient


load_dotenv()
logging.basicConfig(level=logging.WARNING)

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"

In [2]:
RAW_PATH = DATA_DIR / "exports" / "dataset.parquet"
df_raw = pd.read_parquet(RAW_PATH)

## 2. Load Raw Dataset

The input is the Parquet file produced by `01_data_collection.ipynb`:
one row per changed Python file, enriched with patched content, dependency source code,
repository metadata files, file tree, and any human review comments.

## 3. Filter Valid Samples

We drop rows where `patched_content` is missing or empty.
These arise when the GitHub API returned no diff — typically binary files that slipped past
the `.py` extension filter, or enrichment failures for certain edge-case commits.

In [3]:
df = df_raw.dropna(subset=["patched_content"]).copy()
df = df[df["patched_content"].str.strip().astype(bool)].reset_index(drop=True)

dropped = len(df_raw) - len(df)
print(f"Rows before filtering: {len(df_raw):,}")
print(f"Rows dropped (empty patched_content): {dropped:,}")
print(f"Rows after filtering:  {len(df):,}")
print(f"\nUnique repositories: {df['repo'].nunique():,}")
print(f"Unique pull requests: {df.groupby(['repo', 'pr_number']).ngroups:,}")

Rows before filtering: 6,403
Rows dropped (empty patched_content): 392
Rows after filtering:  6,011

Unique repositories: 921
Unique pull requests: 1,326


## 4. Comment Counts

The pipeline routes each sample through one of two prompt variants depending on whether
it has human review comments:

| Condition | Prompt | Output format | Max tokens |
|---|---|---|---:|
| File **has** comments | Classification | `<idx>: <category>` per comment | 256 |
| File has **no** comments | Validation | `true` or `false` | 16 |

In [4]:
def _count_comments(raw) -> int:
    """Count review comments attached to a sample."""
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return 0
    if isinstance(raw, str):
        try:
            return len(json.loads(raw))
        except (json.JSONDecodeError, TypeError):
            return 0
    try:
        return len(list(raw))
    except TypeError:
        return 0


df["n_comments"] = df["comments"].apply(_count_comments)

n_with = (df["n_comments"] > 0).sum()
n_without = (df["n_comments"] == 0).sum()
print(f"Files with review comments:    {n_with:,}  \u2192 classification prompt")
print(f"Files without review comments: {n_without:,}  \u2192 validation prompt")

Files with review comments:    3,521  → classification prompt
Files without review comments: 2,490  → validation prompt


## 5. LLM Client Setup

Credentials are read from environment variables loaded by `python-dotenv`.
The Yandex Cloud endpoint exposes an OpenAI-compatible REST API, so the client wraps
`openai.OpenAI` with a custom `base_url` and `project` (folder ID).

In [5]:
client = LLMClient(
    api_key=os.environ.get("YANDEX_CLOUD_API_KEY", ""),
    folder_id=os.environ.get("YANDEX_CLOUD_FOLDER", ""),
    model="qwen3-235b-a22b-fp8/latest",
)

## 6. Prompt Inspection

We build example prompts for a small subset of samples — one with comments
(classification) and one without (validation) — to verify that context assembly
is correct before launching the full pipeline.

`DataCleaningPipeline` with `llm_client=None` runs only the prompt-building
pass without making any API calls.

In [6]:
# Build prompts for a small subset — no LLM calls needed
preview_pipeline = DataCleaningPipeline(llm_client=None, dep_top_k=5, token_limit=30_000)

sample_clf = df[df["n_comments"] > 0].head(3).copy()
sample_val = df[df["n_comments"] == 0].head(3).copy()

df_clf_prompts, _ = preview_pipeline.build_prompts(sample_clf)
df_val_prompts, _ = preview_pipeline.build_prompts(sample_val)

clf_prompt = df_clf_prompts.iloc[0]["prompt"]
val_prompt = df_val_prompts.iloc[0]["prompt"]

print(f"Classification prompt : {len(clf_prompt):,} chars  (~{len(clf_prompt)//4:,} est. tokens)")
print(f"Validation prompt     : {len(val_prompt):,} chars  (~{len(val_prompt)//4:,} est. tokens)")
print("\n--- Classification prompt (first 600 chars) ---")
print(clf_prompt[:600])

Building prompts: 100%|██████████| 3/3 [00:02<00:00,  1.57it/s]


Classification prompt : 14,832 chars  (~3,708 est. tokens)
Validation prompt     : 12,144 chars  (~3,036 est. tokens)

--- Classification prompt (first 600 chars) ---
You are an expert at analyzing code-review feedback.

You are given a changed file in a pull request together with human review comments.
Classify each comment into exactly one category.

Categories
----------
- blocking_issue  — correctness bugs, security vulnerabilities, data-loss risks, crashes, race conditions
- style           — formatting, naming conventions, import ordering, whitespace
- performance     — efficiency concerns, unnecessary allocations, algorithmic complexity
- best_practice   — design patterns, idiomatic code, architectural suggestions, error-handling improvements
- documenta


## 7. Run Cleaning Pipeline

`DataCleaningPipeline.run()` iterates over all 6,011 samples in a single pass.
For each sample it selects the appropriate prompt variant, calls the judge LLM,
and records the raw response and parsed label.

**Checkpointing** — results are flushed to `.cleaning_checkpoint.json` every 20 rows,
so the pipeline can resume from the last saved position after any interruption.

In [7]:
pipeline = DataCleaningPipeline(
    llm_client=client,
    dep_top_k=5,
    token_limit=30_000,
)

df_cleaned, _ = pipeline.run(
    df,
    checkpoint_path=str(PROJECT_ROOT / ".cleaning_checkpoint.json"),
    checkpoint_every=20,
)

print(f"\nDone. Processed {len(df_cleaned):,} samples.")
print(f"Columns added: {[c for c in df_cleaned.columns if c not in df.columns]}")

Cleaning: 100%|████████████| 6011/6011 [5:00:33<00:00,  3.00s/it]



Done. Processed 6,011 samples.
Columns added: ['has_blocking_issues', 'comment_categories', 'llm_raw_response']


## 8. Results

Samples are split between the two prompt variants:
- **Validation** (`has_blocking_issues`) — files without human comments.
- **Classification** (`comment_categories`) — files with human comments.

Samples exceeding the 30,000-token budget are marked `Too long` and skipped.

In [8]:
print("=== Validation results (files without comments) ===")
validation_rows = df_cleaned[df_cleaned["has_blocking_issues"] != ""]
print(validation_rows["has_blocking_issues"].value_counts())

print("\n=== Classification results (files with comments) ===")
classification_rows = df_cleaned[df_cleaned["comment_categories"] != ""]
print(classification_rows["comment_categories"].value_counts().head(10))

too_long = df_cleaned[
    df_cleaned["has_blocking_issues"].str.startswith("Too long", na=False)
    | df_cleaned["comment_categories"].str.startswith("Too long", na=False)
]
print(f"\nSkipped (too long): {len(too_long):,}")

=== Validation results (files without comments) ===
has_blocking_issues
False                              1897
True                                305
Too long (limit = 30000 tokens)     288
Name: count, dtype: int64

=== Classification results (files with comments) ===
comment_categories
Too long (limit = 30000 tokens)                                         517
['best_practice']                                                       417
['style']                                                               310
['question']                                                            248
['nitpick']                                                             210
['blocking_issue']                                                      115
['documentation']                                                        97
['best_practice', 'best_practice']                                       95
['style', 'style']                                                       92
['best_practice', 'best_p

### Parse-failure analysis

The LLM judge occasionally returns responses that do not match the expected format.
The `_parse_bool` and `_parse_classifications` helpers handle these gracefully:
malformed validation responses default to `False` (treated as clean),
and malformed category tokens default to `other`.

In [9]:
val_mask = df_cleaned["has_blocking_issues"] != ""
clf_mask = df_cleaned["comment_categories"] != ""

print(f"Validation responses stored:    {val_mask.sum():,}")
print(f"Classification responses stored: {clf_mask.sum():,}")

print("\nSample raw responses (validation):")
for i, resp in enumerate(df_cleaned.loc[val_mask, "llm_raw_response"].head(5)):
    print(f"  [{i}] {resp!r}")

print("\nSample raw responses (classification):")
for i, resp in enumerate(df_cleaned.loc[clf_mask, "llm_raw_response"].head(5)):
    print(f"  [{i}] {resp!r}")

Validation responses stored:    2,490
Classification responses stored: 3,521

Sample raw responses (validation):
  [0] 'false'
  [1] 'false'
  [2] 'true'
  [3] 'false'
  [4] 'false'

Sample raw responses (classification):
  [0] '0: best_practice\n1: style'
  [1] '0: nitpick\n1: nitpick'
  [2] '0: blocking_issue'
  [3] '0: best_practice\n1: question'
  [4] '0: style'


## 9. Save Output

In [10]:
OUTPUT_PATH = PROJECT_ROOT / "cleaned_2.parquet"

# Drop the helper column — not part of the schema expected by 03_data_preparation.ipynb
df_out = df_cleaned.drop(columns=["n_comments"], errors="ignore")
df_out.to_parquet(OUTPUT_PATH, index=False, compression="snappy")

size_mb = OUTPUT_PATH.stat().st_size / 1024 / 1024
print(f"Saved {len(df_out):,} rows → {OUTPUT_PATH}")
print(f"File size: {size_mb:.1f} MB")
print(f"\nFinal columns:")
cols = list(df_out.columns)
print(f"  {cols[:6]}")
print(f"   {cols[6:12]}")
print(f"   {cols[12:]}")

Saved 6,011 rows → /Users/nikita/University/DataMining/Project/ai-code-reviewer/cleaned_2.parquet
File size: 124.8 MB

Final columns:
  ['repo', 'pr_number', 'pr_title', 'pr_body', 'repo_star_count', 'commit_sha',
   'path', 'patched_content', 'outgoing_dependencies', 'incoming_dependencies',
   'metadata_files', 'file_tree', 'comments',
   'has_blocking_issues', 'comment_categories', 'llm_raw_response']
